# Structured Output Parsing — Reliable Agent Communication

## What This Notebook Teaches You

Agent systems depend on **structured communication** between the LLM and the execution engine. But LLMs are text generators — they don't natively produce valid JSON, and they often wrap output in prose, markdown, or malformed syntax.

This notebook builds a **complete parsing toolkit** from scratch.

By the end, you will be able to:

1. **Classify parsing failures** — understand the taxonomy of LLM output errors
2. **Build 4 extraction strategies** — regex JSON, JSON repair, XML, and delimiter-based
3. **Create an OutputParser** — auto-selects the best strategy for any input
4. **Implement schema validation** — type checking, required fields, defaults
5. **Build a RetryParser** — feeds errors back to the LLM for self-correction
6. **Measure reliability** — comparative testing across strategies and prompt types

---

> **Prerequisites**: Complete Notebooks 02–03 (Agent Loop, Tool Use).
> **Runtime**: Google Colab with T4 GPU (free tier).
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Structured Output Matters

### 1.1 — The Core Problem

When we ask an LLM to produce JSON, many things can go wrong:

```
PROMPT:  Respond with JSON: {"action": "tool", "tool": "calculator", "arguments": {"expression": "2+2"}}

ACTUAL LLM OUTPUTS (observed in practice):

✓ {"action": "tool", "tool": "calculator", "arguments": {"expression": "2+2"}}
✗ Sure! Here's the JSON: {"action": "tool", "tool": "calculator", "arguments": {"expression": "2+2"}}
✗ ```json
  {"action": "tool", "tool": "calculator", "arguments": {"expression": "2+2"}}
  ```
✗ {'action': 'tool', 'tool': 'calculator', 'arguments': {'expression': '2+2'}}
✗ {action: "tool", tool: "calculator", arguments: {expression: "2+2"}}
✗ {"action": "tool", "tool": "calculator", "arguments": {"expression": "2+2"},}
✗ I'll use the calculator tool with 2+2
```

### 1.2 — Failure Taxonomy

| Category | Example | Frequency |
|----------|---------|-----------|
| **Wrapped in prose** | "Here's my response: {...}" | Very common |
| **Markdown code blocks** | ```json ... ``` | Very common |
| **Single quotes** | {'key': 'value'} | Common |
| **Unquoted keys** | {key: "value"} | Common |
| **Trailing commas** | {"key": "value",} | Common |
| **Missing braces** | "key": "value" | Occasional |
| **Mixed formats** | Action: tool, Name: calculator | Occasional |
| **Complete refusal** | "I can't do that" | Rare |

Each failure mode needs a specific recovery strategy. Let's build them all.

## 2. Strategy 1: Regex JSON Extraction

The simplest approach: find `{...}` blocks in the text. The challenge is handling **nested braces** correctly.

In [ ]:
def extract_json_regex(text: str) -> Optional[Dict[str, Any]]:
    """Extract JSON from text using regex with nested brace handling."""
    text = text.strip()

    # Strategy 1a: Direct parse (fast path)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        pass

    # Strategy 1b: Extract from markdown code blocks
    code_patterns = [
        r'\`\`\`(?:json)?\s*([\s\S]*?)\s*\`\`\`',
        r'\`([^\n\`]*\{[^\n\`]*\}[^\n\`]*)\`',
    ]
    for pattern in code_patterns:
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except (json.JSONDecodeError, TypeError):
                pass

    # Strategy 1c: Find outermost {...} with brace counting
    depth = 0
    start = None
    for i, char in enumerate(text):
        if char == '{':
            if depth == 0:
                start = i
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0 and start is not None:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    pass
                # Don't break — try next {...} block
                start = None

    return None

# Test cases
test_cases = [
    ('Clean JSON', '{"action": "think", "thought": "testing"}'),
    ('Wrapped in prose', 'Here is my response: {"action": "answer", "answer": "42"} Hope that helps!'),
    ('In code block', '```json\n{"action": "tool", "tool": "calc"}\n```'),
    ('Nested braces', '{"action": "tool", "arguments": {"expr": "2+2"}}'),
    ('Multiple blocks', 'First {"bad": true} then {"action": "answer", "answer": "yes"}'),
    ('No JSON', 'I think the answer is forty-two'),
]

print("Strategy 1: Regex JSON Extraction")
print("=" * 60)
for label, text in test_cases:
    result = extract_json_regex(text)
    status = "✓" if result else "✗"
    print(f"  {status} {label}: {result}")

## 3. Strategy 2: JSON Repair

When the LLM produces *almost* valid JSON, we can often fix it. Common repairs:
- Single quotes → double quotes
- Trailing commas → removed
- Unquoted keys → quoted
- Missing closing braces → added

In [ ]:
def repair_json(text: str) -> Optional[Dict[str, Any]]:
    """Attempt to repair malformed JSON and parse it."""
    text = text.strip()

    # First try direct parse
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        pass

    # Extract potential JSON substring first
    # Find the first { and work with everything from there
    brace_start = text.find('{')
    if brace_start == -1:
        return None
    text = text[brace_start:]

    # Find matching closing brace
    depth = 0
    end = len(text)
    for i, char in enumerate(text):
        if char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    text = text[:end]

    # Add missing closing braces
    open_count = text.count('{')
    close_count = text.count('}')
    if open_count > close_count:
        text += '}' * (open_count - close_count)

    # Repair 1: Replace single quotes with double quotes (careful with apostrophes)
    repaired = text
    # Handle single-quoted strings: 'value' -> "value"
    repaired = re.sub(r"(?<=[:,\[{\s])'([^']*)'(?=[,\]}: \n])", r'"\1"', repaired)
    # Handle remaining single quotes around keys
    repaired = re.sub(r"'([^']+)'\s*:", r'"\1":', repaired)
    try:
        return json.loads(repaired)
    except (json.JSONDecodeError, TypeError):
        pass

    # Repair 2: Remove trailing commas
    repaired = re.sub(r',\s*([}\]])', r'\1', repaired)
    try:
        return json.loads(repaired)
    except (json.JSONDecodeError, TypeError):
        pass

    # Repair 3: Quote unquoted keys
    repaired = re.sub(r'(?<=[{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r' "\1":', repaired)
    try:
        return json.loads(repaired)
    except (json.JSONDecodeError, TypeError):
        pass

    # Repair 4: Combined repairs
    combined = text
    combined = re.sub(r"'([^']*)'", r'"\1"', combined)
    combined = re.sub(r',\s*([}\]])', r'\1', combined)
    combined = re.sub(r'(?<=[{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r' "\1":', combined)
    try:
        return json.loads(combined)
    except (json.JSONDecodeError, TypeError):
        pass

    return None

# Test cases for repair
repair_tests = [
    ("Single quotes", "{'action': 'think', 'thought': 'testing'}"),
    ("Trailing comma", '{"action": "answer", "answer": "42",}'),
    ("Unquoted keys", '{action: "think", thought: "testing"}'),
    ("Missing close brace", '{"action": "answer", "answer": "42"'),
    ("Combined issues", "{'action': 'tool', 'tool': 'calc',}"),
    ("Valid JSON", '{"action": "think", "thought": "ok"}'),
    ("No JSON at all", "Just some plain text"),
]

print("Strategy 2: JSON Repair")
print("=" * 60)
for label, text in repair_tests:
    result = repair_json(text)
    status = "✓" if result else "✗"
    print(f"  {status} {label}: {result}")

## 4. Strategy 3: XML Tag Extraction

Some prompt designs use XML-like tags instead of JSON. This can be more reliable because XML tags are less ambiguous than braces.

Format:
```
<action>tool</action>
<tool>calculator</tool>
<arguments>{"expression": "2+2"}</arguments>
```

In [ ]:
def extract_xml_tags(text: str) -> Optional[Dict[str, Any]]:
    """Extract structured data from XML-like tags in text."""
    text = text.strip()

    result = {}

    # Define expected tags
    tag_patterns = [
        ("action", r'<action>(.*?)</action>'),
        ("thought", r'<thought>(.*?)</thought>'),
        ("answer", r'<answer>(.*?)</answer>'),
        ("tool", r'<tool>(.*?)</tool>'),
        ("arguments", r'<arguments>(.*?)</arguments>'),
    ]

    for key, pattern in tag_patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            value = match.group(1).strip()
            # Try to parse JSON values
            if key == "arguments":
                try:
                    value = json.loads(value)
                except (json.JSONDecodeError, TypeError):
                    pass
            result[key] = value

    if "action" in result:
        return result

    return None

# Test cases
xml_tests = [
    ("Full XML", "<action>tool</action>\n<tool>calculator</tool>\n<arguments>{\"expression\": \"2+2\"}</arguments>"),
    ("Think action", "<action>think</action>\n<thought>Let me consider this problem carefully.</thought>"),
    ("Answer action", "<action>answer</action>\n<answer>The result is 42.</answer>"),
    ("Mixed with prose", "I'll respond in XML format:\n<action>tool</action>\n<tool>calc</tool>\nHope that helps!"),
    ("No XML", '{"action": "answer", "answer": "not xml"}'),
]

print("Strategy 3: XML Tag Extraction")
print("=" * 60)
for label, text in xml_tests:
    result = extract_xml_tags(text)
    status = "✓" if result else "✗"
    print(f"  {status} {label}: {result}")

## 5. Strategy 4: Delimiter-Based Extraction

Use unique delimiters that are unlikely to appear in regular text. This is especially useful for free-form content that might contain braces or angle brackets.

In [ ]:
def extract_delimited(text: str, start_delim: str = "<<<", end_delim: str = ">>>") -> Optional[Dict[str, Any]]:
    """Extract structured data from delimiter-marked sections."""
    text = text.strip()
    result = {}

    # Find all delimited sections: <<<KEY>>>content<<<END>>>
    pattern = re.escape(start_delim) + r'(\w+)' + re.escape(end_delim) + r'(.*?)' + re.escape(start_delim) + r'END' + re.escape(end_delim)
    matches = re.findall(pattern, text, re.DOTALL)

    for key, value in matches:
        key = key.lower().strip()
        value = value.strip()
        # Try JSON parsing for structured values
        try:
            value = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            pass
        result[key] = value

    if "action" in result:
        return result

    # Fallback: try key: value format between delimiters
    section_pattern = re.escape(start_delim) + r'(.*?)' + re.escape(end_delim)
    section = re.search(section_pattern, text, re.DOTALL)
    if section:
        content = section.group(1).strip()
        try:
            return json.loads(content)
        except (json.JSONDecodeError, TypeError):
            pass

    return None

# Test cases
delim_tests = [
    ("Standard delimiters", "<<<ACTION>>>tool<<<END>>><<<TOOL>>>calculator<<<END>>><<<ARGUMENTS>>>{\"expression\": \"2+2\"}<<<END>>>"),
    ("Think action", "<<<ACTION>>>think<<<END>>><<<THOUGHT>>>Let me analyze this problem.<<<END>>>"),
    ("JSON in delimiters", "<<<{\"action\": \"answer\", \"answer\": \"42\"}>>>"),
    ("No delimiters", '{"action": "answer", "answer": "no delims"}'),
]

print("Strategy 4: Delimiter Extraction")
print("=" * 60)
for label, text in delim_tests:
    result = extract_delimited(text)
    status = "✓" if result else "✗"
    print(f"  {status} {label}: {result}")

## 6. Building the OutputParser

Now we combine all four strategies into a single parser with automatic strategy selection.

In [ ]:
class OutputParser:
    """Unified parser that tries multiple strategies to extract structured output."""

    STRATEGIES = {
        "json_regex": extract_json_regex,
        "json_repair": repair_json,
        "xml_tags": extract_xml_tags,
        "delimited": extract_delimited,
    }

    # Order to try strategies in auto mode
    AUTO_ORDER = ["json_regex", "json_repair", "xml_tags", "delimited"]

    def __init__(self):
        self.parse_history: List[Dict[str, Any]] = []

    def parse(self, text: str, strategy: str = "auto") -> Tuple[Optional[Dict[str, Any]], str]:
        """Parse text using the specified strategy or auto-detect.

        Returns (parsed_result, strategy_used) or (None, 'failed').
        """
        if strategy != "auto":
            func = self.STRATEGIES.get(strategy)
            if func is None:
                return None, "invalid_strategy"
            result = func(text)
            self._record(text, result, strategy)
            return result, strategy if result else (None, "failed")

        # Auto mode: try each strategy in order
        for strat_name in self.AUTO_ORDER:
            func = self.STRATEGIES[strat_name]
            result = func(text)
            if result is not None:
                self._record(text, result, strat_name)
                return result, strat_name

        self._record(text, None, "failed")
        return None, "failed"

    def _record(self, text: str, result: Optional[Dict], strategy: str):
        """Record parse attempt for analysis."""
        self.parse_history.append({
            "input_length": len(text),
            "success": result is not None,
            "strategy": strategy,
            "result_keys": list(result.keys()) if result else [],
        })

    def stats(self) -> Dict[str, Any]:
        """Return parsing statistics."""
        total = len(self.parse_history)
        if total == 0:
            return {"total": 0}
        successes = sum(1 for h in self.parse_history if h["success"])
        strategy_counts = defaultdict(int)
        for h in self.parse_history:
            if h["success"]:
                strategy_counts[h["strategy"]] += 1
        return {
            "total": total,
            "successes": successes,
            "success_rate": successes / total,
            "strategy_usage": dict(strategy_counts),
        }

# Test the unified parser
parser = OutputParser()

all_tests = [
    '{"action": "think", "thought": "clean json"}',
    'Response: {"action": "answer", "answer": "42"}',
    "{'action': 'think', 'thought': 'single quotes'}",
    '{action: "think", thought: "unquoted keys"}',
    '{"action": "tool", "tool": "calc",}',
    '<action>think</action><thought>xml format</thought>',
    '<<<ACTION>>>answer<<<END>>><<<ANSWER>>>delimiter format<<<END>>>',
    'Just plain text with no structure at all',
]

print("Unified OutputParser Test")
print("=" * 60)
for text in all_tests:
    result, strategy = parser.parse(text)
    status = "✓" if result else "✗"
    print(f"  {status} [{strategy:<12}] {str(result)[:60] if result else 'None'}")

print(f"\nStats: {parser.stats()}")

## 7. Schema Validation from Scratch

Parsing extracts data. Validation ensures it's **correct**. We build a simple schema system.

In [ ]:
@dataclass
class Field:
    """Schema field definition."""
    name: str
    field_type: str  # "string", "int", "float", "bool", "dict", "list"
    required: bool = True
    default: Any = None
    allowed_values: Optional[List[Any]] = None
    description: str = ""

@dataclass
class Schema:
    """Validation schema for structured output."""
    name: str
    fields: List[Field]

    def field_names(self) -> List[str]:
        return [f.name for f in self.fields]

def validate(data: Dict[str, Any], schema: Schema) -> Tuple[bool, List[str], Dict[str, Any]]:
    """Validate data against a schema.

    Returns (is_valid, errors, cleaned_data).
    """
    errors = []
    cleaned = {}

    TYPE_CHECKS = {
        "string": lambda v: isinstance(v, str),
        "int": lambda v: isinstance(v, (int, float)) and (isinstance(v, int) or v == int(v)),
        "float": lambda v: isinstance(v, (int, float)),
        "bool": lambda v: isinstance(v, bool),
        "dict": lambda v: isinstance(v, dict),
        "list": lambda v: isinstance(v, list),
    }

    for f in schema.fields:
        if f.name not in data:
            if f.required:
                if f.default is not None:
                    cleaned[f.name] = f.default
                else:
                    errors.append(f"Missing required field: '{f.name}'")
            else:
                if f.default is not None:
                    cleaned[f.name] = f.default
            continue

        value = data[f.name]

        # Type checking
        checker = TYPE_CHECKS.get(f.field_type)
        if checker and not checker(value):
            # Try type coercion
            try:
                if f.field_type == "string":
                    value = str(value)
                elif f.field_type == "int":
                    value = int(value)
                elif f.field_type == "float":
                    value = float(value)
                elif f.field_type == "bool":
                    value = bool(value)
            except (ValueError, TypeError):
                errors.append(f"Field '{f.name}': expected {f.field_type}, got {type(value).__name__}")
                continue

        # Allowed values check
        if f.allowed_values and value not in f.allowed_values:
            errors.append(f"Field '{f.name}': value '{value}' not in allowed values {f.allowed_values}")
            continue

        cleaned[f.name] = value

    is_valid = len(errors) == 0
    return is_valid, errors, cleaned

print("✓ Schema validation system defined")

## 8. Standard Agent Output Schemas

We define three schemas for the actions our agents produce.

In [ ]:
# Schema for tool call output
ToolCallSchema = Schema(
    name="ToolCallOutput",
    fields=[
        Field("action", "string", required=True, allowed_values=["tool"],
              description="Must be 'tool'"),
        Field("tool", "string", required=True,
              description="Name of the tool to call"),
        Field("arguments", "dict", required=True,
              description="Dictionary of arguments for the tool"),
    ]
)

# Schema for answer output
AnswerSchema = Schema(
    name="AnswerOutput",
    fields=[
        Field("action", "string", required=True, allowed_values=["answer"],
              description="Must be 'answer'"),
        Field("answer", "string", required=True,
              description="The final answer text"),
    ]
)

# Schema for think output
ThinkSchema = Schema(
    name="ThinkOutput",
    fields=[
        Field("action", "string", required=True, allowed_values=["think"],
              description="Must be 'think'"),
        Field("thought", "string", required=True,
              description="The reasoning/thought text"),
    ]
)

# Schema router: pick schema based on action field
def get_schema_for_action(data: Dict[str, Any]) -> Optional[Schema]:
    """Select the appropriate schema based on the action field."""
    action = data.get("action", "")
    schema_map = {
        "tool": ToolCallSchema,
        "answer": AnswerSchema,
        "think": ThinkSchema,
    }
    return schema_map.get(action)

# Test validation
test_data = [
    {"action": "tool", "tool": "calculator", "arguments": {"expression": "2+2"}},
    {"action": "answer", "answer": "The result is 4"},
    {"action": "think", "thought": "Let me consider..."},
    {"action": "tool", "tool": "calc"},  # Missing arguments
    {"action": "invalid", "data": "test"},  # Bad action
    {"action": "answer"},  # Missing answer
]

print("Schema Validation Tests")
print("=" * 60)
for data in test_data:
    schema = get_schema_for_action(data)
    if schema is None:
        print(f"  ✗ No schema for action='{data.get('action')}' | Data: {data}")
        continue
    is_valid, errors, cleaned = validate(data, schema)
    status = "✓" if is_valid else "✗"
    if is_valid:
        print(f"  {status} [{schema.name}] Valid: {cleaned}")
    else:
        print(f"  {status} [{schema.name}] Errors: {errors}")

## 9. Retry-with-Feedback — The RetryParser

When parsing fails, we can feed the error back to the LLM and ask it to fix its output. This is the most powerful recovery strategy.

In [ ]:
class RetryParser:
    """Parser that retries with LLM feedback on failure."""

    def __init__(self, parser: OutputParser = None, max_retries: int = 3):
        self.parser = parser or OutputParser()
        self.max_retries = max_retries
        self.retry_history: List[Dict[str, Any]] = []

    def parse_with_retry(self, messages: List[Dict[str, str]],
                         schema: Schema = None) -> Tuple[Optional[Dict[str, Any]], int]:
        """Parse LLM output with retry on failure.

        Args:
            messages: Current conversation messages
            schema: Optional schema to validate against

        Returns (parsed_result, attempts_used)
        """
        for attempt in range(1, self.max_retries + 1):
            # Generate response
            response = generate(messages, max_new_tokens=512, temperature=0.3)

            # Try to parse
            parsed, strategy = self.parser.parse(response)

            if parsed is None:
                # Parse failed — ask LLM to fix
                error_msg = (
                    f"Your response could not be parsed as valid JSON. "
                    f"Raw output: {response[:200]}\n\n"
                    f"Please respond with ONLY a valid JSON object, no other text. "
                    f"Example: {{"action": "answer", "answer": "your answer here"}}"
                )
                messages = messages + [
                    {"role": "assistant", "content": response},
                    {"role": "user", "content": error_msg},
                ]
                self.retry_history.append({
                    "attempt": attempt, "success": False,
                    "error": "parse_failed", "raw": response[:200]
                })
                continue

            # If schema provided, validate
            if schema is not None:
                action_schema = get_schema_for_action(parsed)
                if action_schema:
                    is_valid, errors, cleaned = validate(parsed, action_schema)
                    if not is_valid:
                        error_msg = (
                            f"Your JSON was parsed but failed validation:\n"
                            f"Errors: {errors}\n\n"
                            f"Please fix these issues and respond with ONLY valid JSON."
                        )
                        messages = messages + [
                            {"role": "assistant", "content": response},
                            {"role": "user", "content": error_msg},
                        ]
                        self.retry_history.append({
                            "attempt": attempt, "success": False,
                            "error": "validation_failed", "errors": errors
                        })
                        continue
                    parsed = cleaned

            self.retry_history.append({
                "attempt": attempt, "success": True, "strategy": strategy
            })
            return parsed, attempt

        return None, self.max_retries

print("✓ RetryParser class defined")

### 9.1 — Testing the RetryParser

In [ ]:
retry_parser = RetryParser(max_retries=3)

# Test with prompts that may produce messy output
test_prompts = [
    "What is 15 * 23? Respond with JSON: {\"action\": \"answer\", \"answer\": \"<result>\"}",
    "Think about the meaning of life. Respond with: {\"action\": \"think\", \"thought\": \"<reasoning>\"}",
    "What day comes after Monday? Answer in JSON format with action and answer fields.",
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt[:80]}...")
    messages = [
        {"role": "system", "content": "Always respond with a valid JSON object. No other text."},
        {"role": "user", "content": prompt},
    ]
    result, attempts = retry_parser.parse_with_retry(messages)
    status = "✓" if result else "✗"
    print(f"  {status} Result: {result}")
    print(f"  Attempts: {attempts}")

print(f"\nRetry history: {len(retry_parser.retry_history)} total attempts")
for h in retry_parser.retry_history:
    s = "✓" if h["success"] else "✗"
    print(f"  {s} Attempt {h['attempt']}: {h.get('strategy', h.get('error', '?'))}")

## 10. Comparative Test — Strategy Reliability

Let's generate 20+ test prompts and measure which parsing strategy succeeds most often on first try.

In [ ]:
# Define test prompts that produce various output formats
test_prompts = [
    "What is 2+2?",
    "What color is the sky?",
    "Name three planets.",
    "What is the capital of France?",
    "How many days in a week?",
    "What is the boiling point of water in Celsius?",
    "Spell 'cat' backwards.",
    "What is the largest ocean?",
    "Name a prime number between 10 and 20.",
    "What language is spoken in Brazil?",
    "How many legs does a spider have?",
    "What is the chemical symbol for gold?",
    "Name the first US president.",
    "What is 100 divided by 4?",
    "How many months have 31 days?",
    "What is the square root of 144?",
    "Name the largest continent.",
    "What color do you get mixing red and blue?",
    "How many sides does a hexagon have?",
    "What year did World War 2 end?",
]

system_prompt = """You are a helpful assistant. Respond with ONLY a JSON object in this format:
{"action": "answer", "answer": "<your answer>"}
No other text, just the JSON."""

# Run all prompts and parse with each strategy
results = {name: {"success": 0, "fail": 0} for name in OutputParser.STRATEGIES}
results["auto"] = {"success": 0, "fail": 0}

all_responses = []

print("Generating responses...")
for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    response = generate(messages, max_new_tokens=256, temperature=0.3)
    all_responses.append(response)
    print(f"  [{i+1}/{len(test_prompts)}] {prompt[:40]}... → {response[:50]}...")

print(f"\nGenerated {len(all_responses)} responses. Now parsing...")
print()

In [ ]:
# Parse each response with each strategy
strategy_results = {name: [] for name in list(OutputParser.STRATEGIES.keys()) + ["auto"]}

for i, response in enumerate(all_responses):
    for strat_name in OutputParser.STRATEGIES:
        func = OutputParser.STRATEGIES[strat_name]
        result = func(response)
        success = result is not None and "action" in result
        strategy_results[strat_name].append(success)

    # Auto mode
    test_parser = OutputParser()
    result, _ = test_parser.parse(response)
    success = result is not None and "action" in result
    strategy_results["auto"].append(success)

# Display results
print("Parsing Strategy Comparison")
print("=" * 60)
print(f"{'Strategy':<20} {'Successes':<12} {'Failures':<12} {'Rate':<10}")
print("-" * 54)

for strat_name in list(OutputParser.STRATEGIES.keys()) + ["auto"]:
    successes = sum(strategy_results[strat_name])
    failures = len(strategy_results[strat_name]) - successes
    rate = successes / len(strategy_results[strat_name]) * 100
    bar = "█" * int(rate / 5) + "░" * (20 - int(rate / 5))
    print(f"  {strat_name:<18} {successes:<12} {failures:<12} {rate:>5.1f}%  {bar}")

print(f"\nTotal prompts tested: {len(test_prompts)}")
best = max(strategy_results, key=lambda k: sum(strategy_results[k]))
print(f"Best strategy: {best} ({sum(strategy_results[best])}/{len(test_prompts)} successes)")

## 11. Putting It All Together — Parse + Validate Pipeline

In [ ]:
def robust_parse(text: str, parser: OutputParser = None) -> Tuple[Optional[Dict[str, Any]], Dict[str, Any]]:
    """Full parsing pipeline: extract → validate → clean.

    Returns (result, metadata) where metadata tracks what happened.
    """
    if parser is None:
        parser = OutputParser()

    metadata = {
        "input_length": len(text),
        "strategy_used": None,
        "validated": False,
        "errors": [],
        "coerced": False,
    }

    # Step 1: Parse
    parsed, strategy = parser.parse(text)
    metadata["strategy_used"] = strategy

    if parsed is None:
        metadata["errors"].append("All parsing strategies failed")
        return None, metadata

    # Step 2: Validate against appropriate schema
    schema = get_schema_for_action(parsed)
    if schema is None:
        # Unknown action — return as-is
        metadata["errors"].append(f"Unknown action: {parsed.get('action')}")
        return parsed, metadata

    is_valid, errors, cleaned = validate(parsed, schema)
    metadata["validated"] = is_valid
    metadata["errors"] = errors
    metadata["coerced"] = cleaned != parsed

    if is_valid:
        return cleaned, metadata
    return parsed, metadata

# Test the full pipeline
pipeline_tests = [
    '{"action": "answer", "answer": "The sky is blue"}',
    "{'action': 'think', 'thought': 'single quotes work'}",
    'Here is my response: {"action": "tool", "tool": "calc", "arguments": {"expression": "2+2"}}',
    '<action>answer</action><answer>XML works too</answer>',
    '{"action": "answer"}',  # Missing answer field
    'No structure at all, just text',
]

print("Full Parse + Validate Pipeline")
print("=" * 60)
for text in pipeline_tests:
    result, meta = robust_parse(text)
    status = "✓" if result and meta["validated"] else "⚠" if result else "✗"
    print(f"  {status} Strategy: {meta['strategy_used']}")
    print(f"    Result: {result}")
    if meta["errors"]:
        print(f"    Errors: {meta['errors']}")
    print()

## 12. Key Takeaways

### What We Built
- **4 Extraction Strategies**: regex JSON, JSON repair, XML tags, delimiters
- **OutputParser**: auto-selecting unified parser
- **Schema Validation**: Field + Schema classes with type checking and coercion
- **3 Agent Output Schemas**: ToolCallOutput, AnswerOutput, ThinkOutput
- **RetryParser**: feeds errors back to the LLM for self-correction
- **Full Pipeline**: parse → validate → clean in one call

### Core Insights

1. **Never trust LLM output format.** Even with clear instructions, LLMs produce malformed JSON 10–30% of the time. Always have fallbacks.

2. **Layer your strategies.** Start with the fastest (direct JSON parse), fall back to increasingly aggressive repair (regex → repair → XML → delimiters).

3. **Separate extraction from validation.** Parsing gets you a dict. Validation ensures it's the *right* dict with the right fields and types.

4. **Retry with feedback works.** When parsing fails, telling the LLM *what went wrong* usually produces a valid response on the next try.

5. **Measure your parsing.** Track success rates by strategy. This tells you whether your prompt engineering or your parsing needs work.

### What's Next
In Notebook 05, we combine everything — the agent loop, tools, and structured parsing — into a **ReAct agent**: the foundational pattern for modern AI agents.

---

```
Notebook 04 Complete — Structured Output Parsing
From messy text to reliable machine-readable decisions
```